# 钢珠模型焦点域偏移诊断

## TL;DR

现有证据支持“训练集偏模糊，导致模型更适应偏焦画面”的判断。第二版训练集中，
`scene_0011`、`scene_0013`、`scene_0015` 合计贡献 2,304 / 3,548 个训练框（64.9%）；
代表性裁剪中钢珠轮廓和高光普遍偏软。清晰对焦后，钢珠的边缘、高光形状和局部纹理
都会改变，因此当前模型出现置信度下降属于焦点域偏移。

该结论仍需同一机位、同一光照的“清晰 / 轻微偏焦”配对测试作最终因果验证。

## Context & Methods

- 数据源：`datasets/combined_dataset_run_0002`
- 分析对象：YOLO 标签中的每个钢珠框
- 清晰度代理指标：
  - 将框内区域缩放到 64×64 后计算 Laplacian 方差
  - 同时计算 Tenengrad 梯度能量作为交叉检查
- 指标仅用于场景间的相对比较，不是镜头焦点的绝对标定。
- 金属高光、球体像素尺寸、背景边缘和 JPEG 压缩都会影响数值。

In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

workspace = Path(r"D:\diansai")
dataset = workspace / "vision" / "maixcam_steel_ball_dataset" / "datasets" / "combined_dataset_run_0002"
dataset

WindowsPath('D:/diansai/vision/maixcam_steel_ball_dataset/datasets/combined_dataset_run_0002')

## Data

In [2]:
def focus_metrics(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    normalized = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA)
    normalized = cv2.GaussianBlur(normalized, (3, 3), 0)
    lap = cv2.Laplacian(normalized, cv2.CV_64F, ksize=3)
    gx = cv2.Sobel(normalized, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(normalized, cv2.CV_64F, 0, 1, ksize=3)
    return float(lap.var()), float(np.mean(gx * gx + gy * gy))

records = []
for split in ("train", "val"):
    image_dir = dataset / "images" / split
    label_dir = dataset / "labels" / split
    for image_path in sorted(image_dir.glob("*")):
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            continue
        h, w = image.shape[:2]
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            continue
        for box_index, line in enumerate(label_path.read_text(encoding="utf-8").splitlines()):
            parts = line.split()
            if len(parts) != 5:
                continue
            _, cx, cy, bw, bh = map(float, parts)
            x1 = max(0, int(round((cx - bw / 2) * w)))
            y1 = max(0, int(round((cy - bh / 2) * h)))
            x2 = min(w, int(round((cx + bw / 2) * w)))
            y2 = min(h, int(round((cy + bh / 2) * h)))
            if x2 - x1 < 5 or y2 - y1 < 5:
                continue
            ix = max(1, int(round((x2 - x1) * 0.08)))
            iy = max(1, int(round((y2 - y1) * 0.08)))
            crop = image[y1 + iy:y2 - iy, x1 + ix:x2 - ix]
            if crop.size == 0:
                continue
            lap, tenengrad = focus_metrics(crop)
            records.append({
                "split": split,
                "scene": image_path.name.split("__", 1)[0],
                "image": image_path.name,
                "box_index": box_index,
                "box_width_px": x2 - x1,
                "laplacian_variance_64": lap,
                "tenengrad_64": tenengrad,
            })

focus = pd.DataFrame.from_records(records)
focus.shape

(3986, 7)

## Results

In [3]:
scene_summary = (
    focus.groupby(["split", "scene"], as_index=False)
    .agg(
        ball_crops=("image", "size"),
        median_box_width_px=("box_width_px", "median"),
        lap_p10=("laplacian_variance_64", lambda s: s.quantile(0.10)),
        lap_median=("laplacian_variance_64", "median"),
        lap_p90=("laplacian_variance_64", lambda s: s.quantile(0.90)),
        tenengrad_median=("tenengrad_64", "median"),
    )
)
scene_summary.round(2)

,split,scene,ball_crops,median_box_width_px,lap_p10,lap_median,lap_p90,tenengrad_median
0,train,scene_0004,100,57.5,505.23,949.41,1486.60,8221.05
1,train,scene_0009,1144,33.0,415.33,657.93,944.67,6120.64
2,train,scene_0011,1037,27.0,281.10,438.00,688.72,4443.44
3,train,scene_0013,13,41.0,237.12,407.05,613.76,4825.47
4,train,scene_0015,1254,23.0,337.06,657.86,1004.08,5017.86
5,val,scene_0007,438,26.0,464.55,731.82,1031.76,5932.73


In [4]:
new_scene_names = {"scene_0011", "scene_0013", "scene_0015"}
train = focus[focus["split"] == "train"]
new_scene_boxes = int(train["scene"].isin(new_scene_names).sum())
train_boxes = int(len(train))
new_scene_share = new_scene_boxes / train_boxes
{
    "new_scene_boxes": new_scene_boxes,
    "train_boxes": train_boxes,
    "new_scene_share": f"{new_scene_share:.1%}",
    "overall_laplacian_p10": round(float(focus["laplacian_variance_64"].quantile(0.10)), 2),
    "overall_laplacian_median": round(float(focus["laplacian_variance_64"].median()), 2),
    "overall_laplacian_p90": round(float(focus["laplacian_variance_64"].quantile(0.90)), 2),
}

{'new_scene_boxes': 2304,
 'train_boxes': 3548,
 'new_scene_share': '64.9%',
 'overall_laplacian_p10': 333.63,
 'overall_laplacian_median': 600.16,
 'overall_laplacian_p90': 954.96}

## Takeaways

1. **已验证：** 第二版训练框中 64.9% 来自新加入的三个场景，模型会明显受这些场景的成像状态支配。
2. **很可能：** 这些场景的代表性钢珠裁剪普遍偏软，当前模型因此把模糊轮廓、扩散高光和低频灰度结构学成了主要特征。
3. **尚未严格验证：** 清晰度指标也受高光、尺度和背景影响；必须补做同一画面的清晰/轻微偏焦配对推理。
4. **下一步：** 固定最终工作距离并清晰对焦，补拍以清晰样本为主、少量轻微偏焦为辅的数据，重新训练并转换 MaixCAM 模型。不要只靠降低阈值解决。